# Language Modeling: N-Gram Smoothing, Causal Transformers & Decoding Strategies

This interactive lab covers:
1. **Statistical N-gram Language Models**: Maximum Likelihood Estimation with Laplace (add-alpha) smoothing.
2. **Intrinsic Evaluation**: Perplexity ($PPL = \exp(\text{Loss})$) as branching factor.
3. **Neural Causal Language Modeling**: Pre-LN GPT-style Transformer with lower-triangular causal self-attention.
4. **Autoregressive Decoding Strategies**: Greedy, Temperature scaling, Top-$k$, and Nucleus (Top-$p$) sampling.

In [ ]:
import sys
from pathlib import Path
for p in [Path.cwd() / 'code', Path('03-nlp/language-modeling/code')]:
    if p.exists():
        sys.path.insert(0, str(p.resolve()))

import torch
from lm_engine import (
    NgramLanguageModel,
    MiniCausalLM,
    compute_perplexity,
    sample_next_token,
    generate_sequence,
)

print('Language Modeling engine loaded successfully!')

## 1. Statistical N-Gram Language Model & Perplexity

We fit bigram and trigram models on a small corpus and compute test sequence perplexity.

In [ ]:
corpus = [
    ['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog'],
    ['the', 'lazy', 'dog', 'barks', 'at', 'the', 'brown', 'fox'],
    ['a', 'quick', 'brown', 'fox', 'runs', 'fast'],
]

bigram_lm = NgramLanguageModel(n=2, alpha=1.0)
bigram_lm.fit(corpus)

test_sentence = ['the', 'quick', 'brown', 'fox']
ppl = bigram_lm.compute_perplexity(test_sentence)
print(f'Test sentence: {" ".join(test_sentence)}')
print(f'Bigram Model Perplexity: {ppl:.2f}')
print(f'P(fox | brown) = {bigram_lm.score("fox", ("brown",)):.4f}')

## 2. Neural Causal Language Model (Pre-LN Transformer)

We instantiate a causal GPT-style model where attention is strictly masked to prevent attending to future tokens.

In [ ]:
torch.manual_seed(42)
vocab_size = 50
lm = MiniCausalLM(vocab_size=vocab_size, hidden_dim=32, num_layers=2, num_heads=4, intermediate_dim=64)

prompt = torch.tensor([[10, 22, 14, 8, 31]])
targets = torch.tensor([[22, 14, 8, 31, 2]])  # Shifted right by 1 position

out = lm(prompt, targets=targets)
loss_val = out['loss'].item()
ppl_val = compute_perplexity(loss_val)

print(f'Forward Logits shape:     {out["logits"].shape}')
print(f'Cross-Entropy Loss:       {loss_val:.4f}')
print(f'Perplexity (exp(Loss)):   {ppl_val:.2f}')

## 3. Decoding & Sampling Strategies

Comparing Greedy vs. Temperature Scaling vs. Top-K vs. Nucleus (Top-P) Sampling on next-token distribution.

In [ ]:
raw_logits = torch.tensor([0.2, 4.5, 3.8, 1.1, -1.0, 0.5])

greedy_tok = sample_next_token(raw_logits, temperature=0.0)
temp_low_tok = sample_next_token(raw_logits, temperature=0.3)
temp_high_tok = sample_next_token(raw_logits, temperature=1.5)
top_k_tok = sample_next_token(raw_logits, temperature=1.0, top_k=2)
top_p_tok = sample_next_token(raw_logits, temperature=1.0, top_p=0.8)

print(f'Greedy (Argmax):               token {greedy_tok}')
print(f'Low Temp (T=0.3 - peaked):     token {temp_low_tok}')
print(f'High Temp (T=1.5 - diverse):   token {temp_high_tok}')
print(f'Top-K (k=2):                   token {top_k_tok}')
print(f'Top-P (p=0.8 nucleus):         token {top_p_tok}')

## 4. Autoregressive Sequence Generation Loop

Generating a continuation sequence given a prompt prefix.

In [ ]:
prompt_tokens = [5, 12, 19]
generated_tokens = generate_sequence(
    model=lm,
    prompt_ids=prompt_tokens,
    max_new_tokens=8,
    temperature=0.8,
    top_k=5,
    top_p=0.9
)

print(f'Initial Prompt:     {prompt_tokens}')
print(f'Generated Sequence: {generated_tokens}')
assert len(generated_tokens) == len(prompt_tokens) + 8